# Домашнее задание по теме «Сложность моделей и ансамбли»

В работе две части: разложение ошибки kNN на синтетических данных и сравнение ансамблей на Wine Quality.

- Оцени $\mathrm{Bias}^2$, $\mathrm{Var}$ и шум для kNN-регрессии при разных $k$.
- Построй валидационные кривые и кривые обучения для диагностики недообучения и переобучения.
- Реализуй блендинг и OOF-стекинг для регрессии.
- Сравни результаты корректной процедуры и обучения метамодели с утечкой.

**Всего 10 баллов.** Ориентировочное время выполнения 3–4 часа.

In [ ]:
!pip install tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone, BaseEstimator, RegressorMixin
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import (train_test_split, KFold, cross_val_predict,
                                     validation_curve, learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm


RANDOM_STATE = 42
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 11})

## Часть I. Сложность модели и разложение ошибки

На семинаре мы разложили тестовую ошибку на три слагаемых:

$$
\mathbb{E}\bigl[(y - \hat{f}(x_0))^{2}\bigr]
= \sigma^{2}_{\varepsilon}
+ \operatorname{Bias}^{2}\bigl[\hat{f}(x_0)\bigr]
+ \operatorname{Var}\bigl[\hat{f}(x_0)\bigr]
$$

Чтобы оценить компоненты численно, генерируем $B$ обучающих выборок, на каждой обучаем модель, складываем предсказания в матрицу $B \times T$, где $T$ число фиксированных точек для прогноза. По столбцам считаем смещение (отклонение среднего от истины) и разброс (дисперсию столбца).

Данные берём те же, что на семинаре, — с известной истинной зависимостью, иначе смещение не посчитать:

$$
y_i = 0.05 x_i^3 - x_i + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}\bigl(0,\ 2.5^2\bigr).
$$
Шум нового отклика независим от обучающей выборки, имеет нулевое среднее и дисперсию $\sigma_\varepsilon^2$. Предсказания имеют конечные вторые моменты. В BVD усреднение идёт по случайности обучения и новому шуму при фиксированном $x_0$.

Здесь `decompose` усредняет по равномерной сетке из $T$ точек и $B$ случайным обучающим выборкам. Столбец `mse` вычислен как сумма оценок и известной дисперсии шума. Это приближение ожидаемой ошибки, а не независимо измеренная MSE на зашумлённом тесте. Случайность Монте-Карло влияет на найденный минимум.


In [ ]:
SIGMA = 2.5                      # Стандартное отклонение шума. Дисперсия = 6.25


def f_true(x):
    """Истинная зависимость."""
    return 0.05 * x ** 3 - x


def make_sample(n, rng):
    """Одна обучающая выборка объёма n."""
    x = rng.uniform(-8, 8, size=n)
    y = f_true(x) + rng.normal(0, SIGMA, size=n)
    return x, y


def decompose(make_model, values, B=300, n=200, n_test=500, seed=RANDOM_STATE):
    """Оценки Bias^2, Var и ожидаемой MSE на фиксированной сетке x."""
    rng_d = np.random.default_rng(seed)
    x_grid = np.linspace(-8, 8, n_test)
    f_grid = f_true(x_grid)
    rows = []

    for v in tqdm(values):
        preds = np.zeros((B, n_test))
        for b in range(B):
            x_b, y_b = make_sample(n, rng_d)
            preds[b] = make_model(v).fit(x_b[:, None], y_b).predict(x_grid[:, None])

        bias2 = ((preds.mean(axis=0) - f_grid) ** 2).mean()
        var = preds.var(axis=0).mean()
        rows.append({'value': v, 'bias2': bias2, 'var': var,
                     'noise': SIGMA ** 2, 'mse': SIGMA ** 2 + bias2 + var})

    return pd.DataFrame(rows)

### Задача 1. Разложение ошибки kNN-регрессии [2 балла]

На семинаре сложность дерева задавалась глубиной: чем больше `max_depth`, тем модель гибче. У метода ближайших соседей роль регулятора играет число соседей $k$ — но работает оно в **обратную** сторону: чем больше $k$, тем модель жёстче.

- Посчитай разложение для `KNeighborsRegressor` при $k \in \{1, 2, 3, 5, 10, 20, 50, 100, 200\}$. Используй готовую `decompose`. [0,5 балла]
- Построй график: по горизонтали $k$, по вертикали — $\mathrm{Bias}^2$, $\mathrm{Var}$, уровень шума и MSE. Ось $k$ сделай логарифмической (`plt.xscale('log')`), отметь оптимум вертикальной линией. [0,5 балла]
- Найди оптимальное $k$ и выпиши для него все три слагаемых. [0,5 балла]
- Ответь на вопросы ниже. [0,5 балла]

**Вопросы.**

1. Чему примерно равен разброс при $k = 1$? Почему он может быть больше дисперсии шума? Учти, что при фиксированном $x_0$ меняются и шум, и положение ближайшего соседа.
2. Что происходит со смещением и разбросом при $k = 200$, если каждая обучающая выборка состоит ровно из 200 наблюдений? Какую модель мы фактически получаем?

In [ ]:
k_values = # your code

result_knn = # your code

plt.plot(result_knn['value'], result_knn['bias2'], marker='o', label=r'$\mathrm{Bias}^2$')
# TODO: also plot Var, noise, MSE

best_knn = result_knn.loc[result_knn['mse'].idxmin()]
plt.axvline(
    # your code
)

# your code: log scale
plt.xlabel('число соседей k (логарифмическая шкала)'); plt.ylabel('вклад в ошибку')
plt.title(
    # your code
    )
plt.legend()
plt.show()

best_val = # your code

print(f"Оптимум: k = {best_val}")
# TODO: print Bias^2, Var, noise, MSE

## Данные

Дальше нам понадобятся реальные данные. Возьмём датасет **Wine Quality**: физико-химические характеристики португальских вин и оценка дегустатора `quality` от 3 до 9. Задача — регрессия: предсказать `quality`.

Датасет состоит из двух файлов — красные и белые вина. Объединим их и добавим признак типа вина.
Точные повторы удаляем до разбиения, чтобы одинаковые строки не попадали одновременно в обучение и оценку. Для сравнения процедур далее используем одну validation. Она не является независимым итоговым тестом после выбора модели по её результатам.


In [ ]:
red_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
white_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv'

df_red = pd.read_csv(red_url, sep=';')
df_white = pd.read_csv(white_url, sep=';')
df_red['type'] = 'red'
df_white['type'] = 'white'

df = pd.concat([df_red, df_white], ignore_index=True)
n_duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f'Удалено точных повторов до разбиения: {n_duplicates}')
df = pd.get_dummies(df, columns=['type'], drop_first=True, dtype='int')

print(f'Форма датасета: {df.shape}')
print(f'Пропусков: {df.isna().sum().sum()}')
df.head(3)

In [ ]:
# Оценки дегустатора распределены очень неравномерно: почти все вина — на 5, 6 и 7.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['quality'], bins=range(2, 11), edgecolor='white', align='left')
axes[0].set_xlabel('quality'); axes[0].set_ylabel('число вин')
axes[0].set_title('Распределение целевой переменной')

df.drop(columns=['quality', 'type_white']).boxplot(ax=axes[1], rot=90)
axes[1].set_yscale('log')
axes[1].set_title('Признаки: масштабы различаются на порядки, есть выбросы')

plt.tight_layout()
plt.show()

print(df['quality'].value_counts().sort_index())

In [ ]:
X = df.drop('quality', axis=1)
y = df['quality']

# Стратификация сохраняет представительство оценок с учётом округления размеров.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
numeric = X.columns.drop('type_white')
# Оставляем сырые признаки. Масштабирование линейных и метрических базисов
# будет обучаться внутри каждой модели, включая holdout и каждый OOF-фолд.
print(f'train: {X_train.shape}, validation: {X_valid.shape}')


### Задача 2. Диагностика недообучения и переобучения [2 балла]

На реальных данных истинная зависимость неизвестна. Кривые обучения помогают диагностировать качество, но не измеряют смещение и разброс из BVD по отдельности.

- Высокие близкие ошибки могут указывать на недообучение. Сначала сравни их с разумным baseline и проверь данные.
- Низкая ошибка на обучении и большой разрыв с CV указывают на переобучение. Разрыв не равен дисперсии предсказаний из BVD.

Задание:

- Построй валидационную кривую дерева регрессии по `max_depth` от 1 до 20 (`validation_curve`, 5 фолдов, метрика `neg_mean_squared_error`). Отметь минимум CV-MSE. Используй только `X_train`, `y_train`. [0,5 балла]
- Построй кривую обучения (`learning_curve`) для дерева с выбранной глубиной и неограниченного дерева. Используй `train_sizes=np.linspace(0.1, 1.0, 8)` и те же 5 фолдов. [1 балл]
- Ответь на вопросы. [0,5 балла]

**Вопросы.**

1. Как называется режим дерева глубины 1, если обе ошибки высоки и близки? Что можно проверить или изменить?
2. У какого дерева разрыв между обучением и CV больше? Как это связано с его гибкостью?
3. Есть ли на кривой признаки того, что дополнительные данные могут помочь неограниченному дереву? Можно ли по конечной кривой гарантировать его качество при бесконечном объёме данных?

Глубина выбрана на этих же CV-фолдах. Кривую выбранного дерева используем для диагностики, а не как независимую итоговую оценку после подбора.


In [ ]:
depth_range = # your code

train_scores, valid_scores = validation_curve(
    # your code
    # DecisionTreeRegressor
    # ...
    )

mse_train_cv = # your code
mse_valid_cv = # your code
best_depth = # your code

plt.plot(depth_range, mse_train_cv, marker='o', label='MSE на обучающих фолдах')
plt.plot(depth_range, mse_valid_cv, marker='o', label='MSE на валидационных фолдах')
plt.axvline(best_depth, color='tab:green', ls=':', lw=2, label=f'оптимум: max_depth = {best_depth}')
plt.xlabel('max_depth'); plt.ylabel('MSE')
plt.title('Валидационная кривая дерева регрессии')
plt.legend()
plt.show()

print(f'Оптимум: max_depth = {best_depth}, CV-MSE = {min(mse_valid_cv):.4f}')
print(f'Дерево глубины 20: CV-MSE = {mse_valid_cv[-1]:.4f}, '
      f'на обучении {mse_train_cv[-1]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

models = [(f'дерево с max_depth = {best_depth}',
           # your code
           ),
          ('неограниченное дерево',
           # your code
           )
          ]

for ax, (title, model) in zip(axes, models):
    sizes, tr, va = learning_curve(
        # your code
        )

    mse_tr, mse_va = # your code
    ax.plot(sizes, mse_tr, marker='o', label='MSE на обучении')
    ax.plot(sizes, mse_va, marker='o', label='MSE на валидации')
    ax.fill_between(sizes, mse_tr, mse_va, alpha=0.15, color='tab:red', label='разрыв')
    ax.set_title(title); ax.set_xlabel('объём обучающей выборки')
    ax.legend(fontsize=9)
    print(f'{title}: разрыв в конце = {mse_va[-1] - mse_tr[-1]:.4f}')

axes[0].set_ylabel('MSE')
plt.tight_layout()
plt.show()

## Часть II. Блендинг и стекинг

Бэггинг и случайный лес усредняют прогнозы однотипных базисов. В блендинге и стекинге комбинацию прогнозов обучает метамодель. Базисы могут быть как одинакового, так и разного типа. Коэффициенты описывают совместную комбинацию и не являются самостоятельной оценкой полезности базиса.

Здесь реализуем блендинг и стекинг для регрессии.

Метамодель обучаем на прогнозах для объектов, которые не участвовали в обучении соответствующих базисов и их препроцессинга. В блендинге используем holdout, в стекинге OOF-прогнозы. В задаче 5 проверим последствия нарушения этой процедуры.

Возьмём шесть базисов: две линейные модели с разной регуляризацией, ядровой метод, метод соседей и два древесных ансамбля.

In [ ]:
def make_base():
    """Свежие базисы с независимым препроцессингом."""
    def scaled(model):
        prep = ColumnTransformer(
            [('numeric', StandardScaler(), list(numeric))], remainder='passthrough')
        return make_pipeline(prep, model)
    return {
        'ridge': scaled(Ridge(alpha=1.0)),
        'lasso': scaled(Lasso(alpha=0.001, max_iter=5000)),
        'svr': scaled(SVR(C=3.0, epsilon=0.1)),
        'knn': scaled(KNeighborsRegressor(n_neighbors=15)),
        'rf': RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        'et': ExtraTreesRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    }

single_scores = {}
for name, model in tqdm(make_base().items()):
    model.fit(X_train, y_train)
    single_scores[name] = mean_squared_error(y_valid, model.predict(X_valid))
pd.Series(single_scores).sort_values().round(4).to_frame('валидационный MSE')


### Задача 3. Свой класс блендинга регрессии [2 балла]

Блендинг работает по схеме с отложенной выборкой:

1. разбить обучающую выборку на две части — `X_tr` и `X_blend`.
2. обучить базовые модели на `X_tr`.
3. получить их предсказания на `X_blend`.
4. на этих предсказаниях обучить метамодель (по умолчанию — линейная регрессия).
5. на валидации использовать те же базовые модели, обученные на `X_tr`, без дообучения.

Реализуй методы `fit()` и `predict()`. [1,5 балла]

Затем обучи блендинг на наших данных и сравни его MSE с лучшим одиночным базисом. Посмотри на веса метамодели — они лежат в `blend.meta_model_.coef_`. [0,5 балла]

In [ ]:
class MyBlendingRegressor(RegressorMixin, BaseEstimator):
    """Блендинг регрессии на отложенной выборке."""

    def __init__(self, base_models: dict, meta_model=None,
                 test_size=0.3, random_state=RANDOM_STATE, shuffle=True):
        self.base_models = base_models
        self.meta_model = meta_model
        self.test_size = test_size
        self.random_state = random_state
        self.shuffle = shuffle

        self.model_order_ = list(base_models.keys())
        self.fitted_base_ = None
        self.meta_model_ = None

    def fit(self, X, y):
        self.model_order_ = list(self.base_models)
        # 1. Отложенная выборка для метамодели
        X_tr, X_blend, y_tr, y_blend = # train_test_split(...)

        # 2. Базовые модели учим только на X_tr
        self.fitted_base_ = # your code
        # NB: каждую модель перед fit оберни в clone(...)

        # 3. Их предсказания на отложенной части — признаки для метамодели
        P_blend = # your code

        # 4. Метамодель учится на предсказаниях, а не на исходных признаках
        self.meta_model_ = # your code

        return self

    def predict(self, X):
        if self.meta_model_ is None:
            raise RuntimeError('Сначала вызови fit.')
        P = # your code
        return self.meta_model_.predict(P)

In [ ]:
blend = # MyBlendingRegressor(...)
# your code: fit
mse_blend = # your code

print(f'Блендинг MSE:            {mse_blend:.4f}')
print(f'Лучший одиночный базис:  {min(single_scores.values()):.4f} '
      f'({min(single_scores, key=single_scores.get)})')

weights_blend = # your code: use blend.meta_model_.coef_

weights_blend.sort_values().plot(kind='barh', color='tab:blue')
plt.axvline(0, color='gray', ls='--')
plt.xlabel('коэффициент метамодели'); plt.title('Коэффициенты блендинга')
plt.tight_layout()
plt.show()

weights_blend.round(3).to_frame('вес')

### Задача 4. Свой класс стекинга регрессии [2,5 балла]

Стекинг — обобщение блендинга. Вместо одной отложенной выборки он использует кросс-валидацию, поэтому метамодель учится на предсказаниях для **всех** обучающих объектов, а не для 30% из них.

Ключевые отличия от блендинга:

- вместо `train_test_split` — `KFold` и out-of-fold предсказания (проще всего через `cross_val_predict`).
- метамодель учится на OOF-матрице `Z_tr` размера (число обучающих объектов × число базисов).
- базовые модели затем **обучаются заново на всей** обучающей выборке — именно они пойдут в инференс.

Реализуй `fit()` и `predict()`. [1,5 балла]

Обучи стекинг, сравни с блендингом и с одиночными моделями. Укажи, у какого метода ниже MSE на этом разбиении. Обсуди, как различия в объёме обучения базисов и метамодели могли повлиять на результат. [1 балл]

In [ ]:
class MyStackingRegressor(RegressorMixin, BaseEstimator):
    """OOF-стекинг для регрессии."""

    def __init__(self, base_models: dict, meta_model=None,
                 cv=5, random_state=RANDOM_STATE):
        self.base_models = base_models
        self.meta_model = meta_model
        self.cv = cv
        self.random_state = random_state

        self.model_order_ = list(base_models.keys())
        self.fitted_base_ = None
        self.meta_model_ = None

    def fit(self, X, y):
        self.model_order_ = list(self.base_models)
        kf = # KFold(...)

        # OOF-матрица: предсказание для объекта делает модель, которая его не видела
        Z_tr = np.zeros((len(y), len(self.model_order_)))
        for j, name in enumerate(self.model_order_):
            Z_tr[:, j] = # cross_val_predict(...)

        # Сохрани Z_tr в self.oof_predictions_ для диагностики корреляций.
        # Метамодель учится на OOF-предсказаниях
        self.meta_model_ = # your code

        # Базисы обучаем заново на всей выборке — они пойдут в инференс
        self.fitted_base_ = # your code

        return self

    def predict(self, X):
        if self.meta_model_ is None:
            raise RuntimeError('Сначала вызови fit.')
        P = # your code
        return self.meta_model_.predict(P)

In [ ]:
stack = # MyStackingRegressor(...)
# your code: fit
mse_stack = # your code

comparison = pd.Series({**single_scores,
                        'БЛЕНДИНГ': mse_blend,
                        'СТЕКИНГ': mse_stack}).sort_values()

colors = ['tab:red' if n in ('БЛЕНДИНГ', 'СТЕКИНГ') else 'tab:blue' for n in comparison.index]
comparison.plot(kind='barh', color=colors)
plt.xlabel('валидационный MSE'); plt.title('Одиночные модели против ансамблей')
plt.tight_layout()
plt.show()

comparison.round(4).to_frame('валидационный MSE')

### Задача 5. Обучение метамодели с утечкой [1,5 балла]

Проверим обучение метамодели на прогнозах для тех же объектов, на которых обучались базисы.

- Напиши «наивный» стекинг: обучи базовые модели на всей обучающей выборке, получи их предсказания **на ней же** и на этих предсказаниях обучи метамодель. [0,5 балла]
- Сравни три числа: MSE наивного стекинга на **обучающей** выборке, его же MSE на валидации и MSE OOF-стекинга на валидации. [0,5 балла]
- Сравни веса метамодели у наивного и OOF-стекинга. [0,25 балла]
- Ответь на вопрос. [0,25 балла]

**Вопрос.** Как коэффициенты метамодели связаны с качеством базисов на train? Почему качество на train отличается от качества на validation?

In [ ]:
# Наивный стекинг: метамодель учится на предсказаниях по тем же данным,
# на которых обучались базисы.

naive_base = # your code
Z_naive = # your code
meta_naive = # your code

P_valid_naive = # your code

mse_naive_train = # your code
mse_naive_valid = # your code

print(f'Наивный стекинг на ОБУЧЕНИИ: {mse_naive_train:.4f}')
print(f'Наивный стекинг на VALIDATION:    {mse_naive_valid:.4f}')
print(f'OOF-стекинг на валидации: {mse_stack:.4f}')

weights = pd.DataFrame({
    'наивный': pd.Series(meta_naive.coef_, index=list(naive_base.keys())),
    'OOF': pd.Series(stack.meta_model_.coef_, index=stack.model_order_),
})

weights.plot(kind='barh')
plt.axvline(0, color='gray', ls='--')
plt.xlabel('вес в метамодели'); plt.title('Веса метамодели: с утечкой и без')
plt.tight_layout()
plt.show()

print(weights.round(3))

## Что дальше

На Неделе 8 мы разберём случайный лес, его гиперпараметры, OOB-оценку и важность признаков. Затем перейдём к бустингу: его базисы обучаются по очереди, каждый следующий учитывает ошибки предыдущих. Воздействие ансамбля на смещение и разброс зависит от базовых моделей, данных и параметров.


## Что отработали в домашнем задании

- Оценку смещения и разброса kNN на синтетических данных.
- Диагностику недообучения и переобучения по валидационным кривым и кривым обучения.
- Реализацию блендинга и OOF-стекинга для регрессии.
- Сравнение обучения метамодели по OOF-прогнозам и по прогнозам с утечкой.